# FDTD境界条件の違いについての詳細解説
本ノートブックでは **kukei1_ori.py** と **kukei2_plus_animation.py** の
大きな構造的違いである
- **Uy境界条件の適用回数・タイミングの違い**
- **きず内部の応力 (T1,T3,T5) を0にするタイミングの違い**

について詳細にまとめます。

## 1. 更新ステップの流れ比較
### kukei1_ori.py の 1ステップ
1. 入射波添加
2. **応力境界条件**
3. **Uy境界条件 (1回目)**
4. **きず内 T1,T3,T5 = 0（1回目）**
5. **Uy境界条件 (2回目)**
6. 応力更新（T1, T3, T5）
7. **きず内 T1,T3,T5 = 0（2回目）**
8. **Uy境界条件 (3回目)**
9. 粒子速度 Ux,Uy 更新

**→ Uy BC 3回、きず応力ゼロ 2回**

### kukei2_plus_animation.py の 1ステップ
1. 入射波添加
2. 応力境界条件
3. **Uy境界条件（1回のみ）**
4. 応力更新（T1, T3, T5）
5. **きず内 T1,T3,T5 = 0（1回のみ）**
6. 粒子速度 Ux,Uy 更新

**→ Uy BC 1回、きず応力ゼロ 1回**

## 2. Uy境界条件の数値的意味
Uy境界条件は、自由境界（せん断応力ゼロ）条件を満たすために、
境界上の粒子速度 Uy を **T3, T5 の値に応じて補正**します。

一般形は：
```python
Uy[x,0]  = Uy[x,0]  - (4/rho) * dt/dx * T3[x,0]
Uy[x,ny] = Uy[x,ny] - (4/rho) * dt/dx * (-T3[x,ny-1])
Uy[nx,y] = Uy[nx,y] - (4/rho) * dt/dx * (-T5[nx-1,y])
```

### 適用回数の物理的効果
#### kukei1_ori.py（3回）
- 境界でせん断応力ゼロが **強制的に何度も補正される**。
- → 境界付近の挙動が **より“固く”なる（動きづらい）**。
- → 高周波成分（波長が短く、離散化の影響を受けやすい）が特に影響を受ける。

#### kukei2_plus_animation.py（1回）
- 一般的なFDTDの境界処理。
- → 自由表面が自然に近い振る舞いになりやすい。
- → ori版より境界反射やレイリー波が少し変わる。

## 3. きず領域の応力ゼロ化の違い
きず（溝）内部は自由表面に相当するため、
**T1 = T3 = T5 = 0** を適用する必要があります。

しかし、この 0 クリアを「**いつ行うか**」が両者で異なります。

### kukei1_ori.py（2回）
- 応力更新 *前* に 1回
- 応力更新 *後* に 1回

これにより：
- きず内部に **一瞬たりとも応力が生じない** よう強く拘束
- きず先端の応力集中挙動が **過剰に抑制** される可能性
- 波の反射・散乱の高周波成分が変化しやすい

### kukei2_plus_animation.py（1回）
- 応力更新後に1回だけゼロ化

これにより：
- 更新計算の途中できず内に一瞬応力が立つが、その後きれいに0に戻すだけ
- → 数値的には **自然なマスク方式**
- → きず先端の応力集中や散乱の計算が素直になる

## 4. 高周波成分への影響
- **Uy境界条件が多い（ori版）** → 短波長（高周波）の反射が強くなる傾向
- **きずマスクが強い（ori版）** → きず先端の局所挙動がやや人工的
- → FFTで見ると：
  - 高周波領域のピーク位置
  - 高周波ノイズの量
  - 位相のズレ
  がわずかに変わりうる

特に 5〜10 MHz 付近はメッシュ依存も強く、境界処理差が現れやすいです。

## 5. まとめ
| 項目 | kukei1_ori.py | kukei2_plus_animation.py |
|------|----------------|---------------------------|
| Uy境界条件 | **3回** | **1回** |
| きず応力ゼロ | **2回** | **1回** |
| 境界の“固さ” | 強い | 弱い（自然） |
| 高周波反射 | やや大きい | 控えめ |
| きず先端挙動 | 抑制されやすい | 素直に出る |

**→ 両コードは境界処理がかなり違うため、FFTや高周波挙動にも影響が出る** という結論になります。